In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import norm
from scipy.optimize import brentq
import matplotlib.pyplot as plt

In [ ]:
# ================================================================
# HELPER FUNCTIONS
# ================================================================

def norm_cdf(x): return norm.cdf(x)
def norm_ppf(x): return norm.ppf(x)

LGD = 0.60        # ISDA standard for sovereign CDS
T   = 5.0         # 5-year horizon (matches 5Y CDS)


def pd_to_cds_spread(pd_val, T=T, lgd=LGD):
    """PD → approximate CDS spread in bps."""
    pd_c = np.clip(pd_val, 1e-12, 1 - 1e-6)
    return -(1.0 / T) * np.log(1.0 - lgd * pd_c) * 10000


def merton_pd_cf(V0, B0, sigma, mu, T=T):
    """Closed-form Merton PD and DD."""
    denom = np.maximum(sigma * np.sqrt(T), 1e-10)
    dd = (np.log(V0 / B0) + (mu - 0.5 * sigma**2) * T) / denom
    pd = norm.cdf(-dd)
    return pd, dd


# ================================================================
# ROLLING CALIBRATION
# ================================================================

def _calibrate_window(B0, sigma, mu, cds_obs, T=T, lgd=LGD,
                      scale_bounds=(1.01, 10.0)):
    """
    Find scale_mult for a window so M1's avg implied spread
    matches the avg observed CDS spread.
    Returns (scale, status_str).
    """
    valid = (~np.isnan(B0) & ~np.isnan(sigma) & ~np.isnan(mu) &
             ~np.isnan(cds_obs) & (B0 > 0) & (sigma > 0) & (cds_obs > 0))
    B0_v, sig_v, mu_v, cds_v = B0[valid], sigma[valid], mu[valid], cds_obs[valid]

    if len(B0_v) < 10:
        return None, 'insufficient_data'

    target = np.mean(cds_v)

    def err(s):
        pd_vals, _ = merton_pd_cf(s * B0_v, B0_v, sig_v, mu_v, T)
        return np.mean(pd_to_cds_spread(pd_vals, T, lgd)) - target

    try:
        e_lo, e_hi = err(scale_bounds[0]), err(scale_bounds[1])
    except Exception:
        return None, 'eval_failed'

    if e_lo < 0:
        return scale_bounds[0], 'at_lower_bound'
    if e_hi > 0:
        return scale_bounds[1], 'at_upper_bound'

    try:
        return brentq(err, scale_bounds[0], scale_bounds[1],
                      xtol=1e-4, maxiter=100), 'converged'
    except Exception:
        return None, 'brentq_failed'


def calibrate_rolling(df, cds_col='cds_spread',
                      barrier_col='default_barrier',
                      sigma_col='msci_vol_annual',
                      country_col='country_clean',
                      mu_mode='zero',
                      window_weeks=52,
                      min_obs=26,
                      T=T, lgd=LGD,
                      scale_bounds=(1.01, 10.0)):
    """
    Rolling calibration: at each week t, find scale_mult using
    the trailing window so M1's avg implied spread ≈ avg CDS.
    
    Handles structural breaks naturally.
    """
    df_out = df.copy()
    df_out['scale_mult'] = np.nan

    print(f'Rolling calibration: window={window_weeks}w, '
          f'min_obs={min_obs}, LGD={lgd}')

    for country in sorted(df_out[country_col].unique()):
        mask = df_out[country_col] == country
        idx = df_out.loc[mask].index

        B0_all  = df_out.loc[mask, barrier_col].values
        sig_all = df_out.loc[mask, sigma_col].values
        cds_all = df_out.loc[mask, cds_col].values
        mu_all  = np.zeros(len(idx))

        n = len(idx)
        last_good = None
        conv = 0

        for t in range(n):
            start = max(0, t - window_weeks + 1)
            if t - start + 1 < min_obs:
                if last_good is not None:
                    df_out.loc[idx[t], 'scale_mult'] = last_good
                continue

            scale, status = _calibrate_window(
                B0_all[start:t+1], sig_all[start:t+1],
                mu_all[start:t+1], cds_all[start:t+1],
                T, lgd, scale_bounds)

            if scale is not None:
                df_out.loc[idx[t], 'scale_mult'] = scale
                last_good = scale
                if status == 'converged': conv += 1
            elif last_good is not None:
                df_out.loc[idx[t], 'scale_mult'] = last_good

        # Fill edges
        s = df_out.loc[mask, 'scale_mult']
        df_out.loc[mask, 'scale_mult'] = s.ffill().bfill()

        scales = df_out.loc[mask, 'scale_mult'].dropna()
        print(f'  {country:<20s}  scale: {scales.mean():.2f} '
              f'[{scales.min():.2f}–{scales.max():.2f}]  conv={conv}/{n}')

    return df_out

In [ ]:
# ================================================================
# MONTE CARLO JUMP DIFFUSION (Negative-Only, Stochastic Size)
# ================================================================

def mc_pd_dd_jump_stochastic(V0, B0, sigma, mu, lam, theta, delta,
                             T=T, n_paths=20000, seed=7):
    """
    Monte Carlo PD with negative-only jump-diffusion.
    
    lam:   jump intensity per year (vector)
    theta: mean of log-jump size, should be < 0 (vector)
    delta: std dev of log-jump size (vector)
    """
    rng = np.random.default_rng(seed)
    n_obs = len(V0)

    drift = (mu - 0.5 * sigma**2) * T
    volT  = sigma * np.sqrt(T)

    # 1. Poisson jump counts
    N = rng.poisson(lam[:, None] * T, size=(n_obs, n_paths))

    # 2. Aggregate jump effect (sum of N normal draws)
    agg_mean = N * theta[:, None]
    agg_std  = np.sqrt(np.maximum(N, 0)) * delta[:, None]
    Z_jump = rng.standard_normal((n_obs, n_paths))

    # Clamp to non-positive (negative-only jumps)
    total_jump = np.minimum(agg_mean + agg_std * Z_jump, 0.0)

    # 3. Diffusion
    Z_diff = rng.standard_normal((n_obs, n_paths))

    # 4. Terminal log asset value
    logVT = (np.log(V0)[:, None] + drift[:, None]
             + volT[:, None] * Z_diff + total_jump)

    # Metrics
    PD = (logVT < np.log(B0)[:, None]).mean(axis=1)
    DD = norm_ppf(1 - np.clip(PD, 1e-10, 1 - 1e-10))
    SE = np.sqrt(PD * (1 - PD) / n_paths)

    return PD, DD, SE


# ================================================================
# MAIN RUNNER: Calibrate + Run M1, M2, M3
# ================================================================

def run_all_models(panel_csv, out_csv,
                   mu_mode='zero',
                   window_weeks=52,
                   ovx_thresholds=(30, 50),
                   ovx_lambdas=(0.01, 0.05, 0.20),
                   n_paths=50000):
    """
    Full pipeline:
      1. Load panel
      2. Rolling-calibrate V0/B0 per country (M1 → trailing CDS)
      3. Run M1 (closed-form), M2 (constant λ), M3 (OVX-regime λ)
      4. Convert PDs to implied CDS spreads
    """
    df = pd.read_csv(panel_csv, parse_dates=['date'])

    # --- Oil exporter classification ---
    OIL_EXPORTERS = ['Saudi Arabia', 'United Arab Emirates', 'Qatar',
                     'Colombia', 'Mexico', 'Brazil', 'Egypt', 'Malaysia']
    CONTROLS      = ['Indonesia', 'Philippines', 'Turkey', 'Chile',
                     'China', 'South Africa', 'South Korea', 'Thailand']
    _oil_set  = {c.lower() for c in OIL_EXPORTERS}
    _ctrl_set = {c.lower() for c in CONTROLS}
    df['oil_exporter'] = df['country_clean'].str.lower().apply(
        lambda x: 1 if x in _oil_set else (0 if x in _ctrl_set else np.nan))
    n_oil  = (df['oil_exporter'] == 1).sum()
    n_ctrl = (df['oil_exporter'] == 0).sum()
    n_oth  = df['oil_exporter'].isna().sum()
    print(f'  Groups: {n_oil} oil-exporter rows, {n_ctrl} control rows, {n_oth} other')
    if n_oth > 0:
        unmatched = df.loc[df['oil_exporter'].isna(), 'country_clean'].unique()
        print(f'  Unmatched countries: {list(unmatched)}')

    # --- Drift ---
    if mu_mode == 'hist':
        df['mu_hist'] = (
            df.groupby('country_clean')['msci_ret_weekly']
            .transform(lambda x: x.rolling(52, min_periods=10)
                       .mean().shift(1) * 52)
            .fillna(0)
        )
        mu = df['mu_hist'].values
    else:
        mu = np.zeros(len(df))

    # --- Rolling calibration of V0/B0 ---
    print('=' * 60)
    print('  STEP 1: Rolling Calibration (M1 → observed CDS)')
    print('=' * 60)
    df = calibrate_rolling(
        df, cds_col='cds_spread',
        window_weeks=window_weeks,
    )

    # Drop rows where calibration failed
    n_before = len(df)
    df = df.dropna(subset=['scale_mult']).copy()
    print(f'\n  Kept {len(df)}/{n_before} rows with valid scale')

    # --- Build arrays ---
    sigma = df['msci_vol_annual'].values
    B0    = df['default_barrier'].values
    V0    = df['scale_mult'].values * B0   # <-- calibrated!
    theta = df['theta'].values
    delta = df['delta'].values
    ovx   = df['OVX'].values

    if mu_mode == 'hist':
        mu = df['mu_hist'].values
    else:
        mu = np.zeros(len(df))

    # ---- M1: Baseline Merton (closed-form) ----
    print('\n' + '=' * 60)
    print('  STEP 2: Running Models')
    print('=' * 60)

    pd_m1, dd_m1 = merton_pd_cf(V0, B0, sigma, mu)

    # ---- M3 lambda: OVX regime-based ----
    low_t, high_t = ovx_thresholds
    lam_calm, lam_elev, lam_stress = ovx_lambdas

    lam_ovx = np.where(
        ovx < low_t,  lam_calm,
        np.where(ovx < high_t, lam_elev, lam_stress)
    )

    # ---- M2 lambda: constant = unconditional avg of M3 ----
    lambda_const = np.mean(lam_ovx)
    print(f'  M2 λ_const (from M3 avg): {lambda_const:.4f}')
    lam_const_arr = np.full(len(df), lambda_const)

    # ---- M2: Constant jump (MC) ----
    print('  Running M2 (constant λ)...')
    pd_m2, dd_m2, _ = mc_pd_dd_jump_stochastic(
        V0, B0, sigma, mu, lam_const_arr, theta, delta,
        n_paths=n_paths, seed=11)

    # ---- M3: OVX-regime jump (MC) ----
    print('  Running M3 (OVX-regime λ)...')
    pd_m3, dd_m3, _ = mc_pd_dd_jump_stochastic(
        V0, B0, sigma, mu, lam_ovx, theta, delta,
        n_paths=n_paths, seed=22)

    # ---- Store results ----
    df['pd_cf_gbm']        = pd_m1
    df['dd_cf_gbm']        = dd_m1
    df['pd_mc_jump_const'] = pd_m2
    df['dd_mc_jump_const'] = dd_m2
    df['pd_mc_jump_ovx']   = pd_m3
    df['dd_mc_jump_ovx']   = dd_m3

    # ---- Implied CDS spreads (bps) ----
    df['spread_m1'] = pd_to_cds_spread(pd_m1)
    df['spread_m2'] = pd_to_cds_spread(pd_m2)
    df['spread_m3'] = pd_to_cds_spread(pd_m3)

    # ---- Summary ----
    print(f'\n{"="*60}')
    print('  MODEL SUMMARY')
    print('=' * 60)
    obs = df['cds_spread']
    for label, col in [('M1 (GBM)', 'spread_m1'),
                       ('M2 (const λ)', 'spread_m2'),
                       ('M3 (OVX λ)',  'spread_m3')]:
        impl = df[col]
        corr = obs.corr(impl)
        rmse = np.sqrt(np.mean((obs - impl)**2))
        print(f'  {label:<15s}  mean={impl.mean():7.0f}bps  '
              f'corr={corr:.3f}  rmse={rmse:.0f}bps')
    print(f'  {"Observed":<15s}  mean={obs.mean():7.0f}bps')

    df.to_csv(out_csv, index=False)
    print(f'\n  Saved to {out_csv}')
    return df

In [ ]:
in_path  = 'data/processed/structural_model_panel_built.csv'
out_path = 'output/mc_calibrated_results.csv'

df_res = run_all_models(
    panel_csv=in_path,
    out_csv=out_path,
    mu_mode='zero',
    window_weeks=52,               # 1-year rolling calibration
    ovx_thresholds=(30, 50),        # calm < 30, elevated 30-50, stress > 50
    ovx_lambdas=(0.01, 0.05, 0.20), # jump intensity per regime
    n_paths=10000,                  # increase to 50000 for final run
)

In [ ]:
country = 'brazil'   # <-- change here

df_c = (df_res[df_res['country_clean'].str.lower() == country.lower()]
        .sort_values('date').copy())

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# --- Top panel: Distance to Default ---
ax = axes[0]
ax.plot(df_c['date'], df_c['dd_cf_gbm'],        label='M1: GBM (closed-form)', lw=2)
ax.plot(df_c['date'], df_c['dd_mc_jump_const'],  label='M2: Jump (constant λ)', alpha=0.8)
ax.plot(df_c['date'], df_c['dd_mc_jump_ovx'],    label='M3: Jump (OVX λ)',      alpha=0.8)
ax.axhline(0, color='black', lw=1, ls=':')
ax.set_ylabel('Distance to Default')
ax.set_title(f'Distance to Default — {country.title()}')
ax.legend()
ax.grid(True, alpha=0.3)

# --- Bottom panel: Implied spread vs observed CDS ---
ax = axes[1]
ax.plot(df_c['date'], df_c['cds_spread'], label='Observed CDS', color='black', lw=2)
ax.plot(df_c['date'], df_c['spread_m1'],  label='M1: GBM',          alpha=0.8)
ax.plot(df_c['date'], df_c['spread_m2'],  label='M2: Jump (const)', alpha=0.8)
ax.plot(df_c['date'], df_c['spread_m3'],  label='M3: Jump (OVX)',   alpha=0.8)
ax.set_ylabel('CDS Spread (bps)')
ax.set_xlabel('Date')
ax.set_title(f'Implied vs Observed CDS Spread — {country.title()}')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Calibration diagnostic: how does scale_mult evolve?
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

ax = axes[0]
ax.plot(df_c['date'], df_c['scale_mult'], color='tab:purple', lw=2)
ax.set_ylabel('Calibrated Scale (V0 / B0)')
ax.set_title(f'Rolling Scale Calibration — {country.title()}')
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(df_c['date'], df_c['cds_spread'], label='Observed CDS', color='black', lw=1.5)
ax2 = ax.twinx()
ax2.plot(df_c['date'], df_c['OVX'], label='OVX', color='tab:red', alpha=0.5)
ax.set_ylabel('CDS Spread (bps)')
ax2.set_ylabel('OVX', color='tab:red')
ax.set_title(f'CDS Spread & OVX — {country.title()}')
ax.legend(loc='upper left')
ax2.legend(loc='upper right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
import statsmodels.api as sm

def run_reg(x, y):
    X = sm.add_constant(x)
    model = sm.OLS(y, X, missing='drop').fit(cov_type='HC1')
    return model.rsquared, model.params

obs_cds = df_c['cds_spread']

# Compare implied spreads directly against observed CDS
r2_m1, _ = run_reg(df_c['spread_m1'], obs_cds)
r2_m2, _ = run_reg(df_c['spread_m2'], obs_cds)
r2_m3, _ = run_reg(df_c['spread_m3'], obs_cds)

print(f'R² Comparison — {country.title()}')
print(f'  M1 (GBM):           {r2_m1:.4f}')
print(f'  M2 (Jump const λ):  {r2_m2:.4f}')
print(f'  M3 (Jump OVX λ):    {r2_m3:.4f}')

# RMSE
for label, col in [('M1', 'spread_m1'), ('M2', 'spread_m2'), ('M3', 'spread_m3')]:
    rmse = np.sqrt(np.mean((obs_cds - df_c[col])**2))
    corr = obs_cds.corr(df_c[col])
    print(f'  {label}  RMSE={rmse:.0f}bps  Corr={corr:.3f}')

In [ ]:
# ================================================================
# CHANGE-ON-CHANGE REGRESSIONS
# ================================================================
# Why changes? The *level* of implied spread depends on the
# calibrated scale (V0/B0), which is a free parameter.
# But *changes* in implied spread reflect model dynamics:
# does the model move in the right direction, at the right time,
# by the right amount? This is what we actually want to test.
#
# ΔS_obs(t) = α + β × ΔS_model(t) + ε(t)
#
# β ≈ 1 means model changes track observed changes 1:1
# R² tells us how much of the *variation* in CDS moves is explained
# ================================================================

import statsmodels.api as sm


def change_on_change_reg(df_country, cds_col='cds_spread',
                         model_cols=None, country_label=''):
    """
    Regress weekly changes in observed CDS on weekly changes
    in model-implied CDS. Reports R², β, and Newey-West t-stats.
    """
    if model_cols is None:
        model_cols = {
            'M1 (GBM)':       'spread_m1',
            'M2 (const λ)':   'spread_m2',
            'M3 (OVX λ)':     'spread_m3',
        }

    dc = df_country.sort_values('date').copy()
    dc['d_cds'] = dc[cds_col].diff()

    results = []
    for label, col in model_cols.items():
        dc[f'd_{col}'] = dc[col].diff()

        tmp = dc[['d_cds', f'd_{col}']].dropna()
        if len(tmp) < 20:
            continue

        y = tmp['d_cds'].values
        X = sm.add_constant(tmp[f'd_{col}'].values)

        # Newey-West HAC standard errors (4 lags for weekly data)
        reg = sm.OLS(y, X).fit(cov_type='HAC',
                              cov_kwds={'maxlags': 4})

        results.append({
            'model': label,
            'R2': reg.rsquared,
            'beta': reg.params[1],
            't_stat': reg.tvalues[1],
            'p_value': reg.pvalues[1],
            'alpha': reg.params[0],
            'n_obs': int(reg.nobs),
        })

    res_df = pd.DataFrame(results)

    if country_label:
        print(f'\nΔCDS ~ ΔModel Spread — {country_label}')
    print('-' * 70)
    print(f'{"Model":<16s} {"R²":>8s} {"β":>8s} {"t-stat":>8s} '
          f'{"p-val":>8s} {"α":>8s} {"N":>6s}')
    print('-' * 70)
    for _, r in res_df.iterrows():
        sig = '***' if r['p_value'] < 0.01 else \
              '**'  if r['p_value'] < 0.05 else \
              '*'   if r['p_value'] < 0.10 else ''
        print(f'{r["model"]:<16s} {r["R2"]:8.4f} {r["beta"]:8.3f} '
              f'{r["t_stat"]:8.2f}{sig:<3s} '
              f'{r["p_value"]:8.4f} {r["alpha"]:8.2f} {r["n_obs"]:6d}')
    print('-' * 70)

    return res_df


# --- Run for the selected country ---
res_country = change_on_change_reg(df_c, country_label=country.title())

In [ ]:
# ================================================================
# PANEL: Change-on-Change R² by Country + DiD Test
# ================================================================
# For each country: regress ΔCDS on ΔModel_spread, store R².
# Then test: is R² improvement from M1→M3 larger for oil exporters?
# ================================================================

panel_rows = []
for ctry, grp in df_res.groupby('country_clean'):
    oil = grp['oil_exporter'].iloc[0] if 'oil_exporter' in grp.columns else np.nan

    grp = grp.sort_values('date').copy()
    grp['d_cds'] = grp['cds_spread'].diff()

    row = {'country': ctry, 'oil_exporter': oil}

    for m, col in [('m1','spread_m1'),('m2','spread_m2'),('m3','spread_m3')]:
        grp[f'd_{col}'] = grp[col].diff()
        tmp = grp[['d_cds', f'd_{col}']].dropna()
        if len(tmp) < 30:
            continue
        y = tmp['d_cds'].values
        X = sm.add_constant(tmp[f'd_{col}'].values)
        reg = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 4})
        row[f'R2_{m}']   = reg.rsquared
        row[f'beta_{m}'] = reg.params[1]

    panel_rows.append(row)

r2_panel = pd.DataFrame(panel_rows)

# Improvements
r2_panel['R2_gain_m3_vs_m1'] = r2_panel['R2_m3'] - r2_panel['R2_m1']
r2_panel['R2_gain_m3_vs_m2'] = r2_panel['R2_m3'] - r2_panel['R2_m2']

# Display
r2_sorted = r2_panel.sort_values('oil_exporter', ascending=False)
print('Change-on-Change R² by Country')
print('=' * 85)
print(f'{"Country":<20s} {"Oil":>3s} {"R²_M1":>7s} {"R²_M2":>7s} '
      f'{"R²_M3":>7s} {"Δ(M3-M1)":>9s} {"Δ(M3-M2)":>9s}')
print('-' * 85)
for _, r in r2_sorted.iterrows():
    print(f'{r["country"]:<20s} {int(r["oil_exporter"]):>3d} '
          f'{r.get("R2_m1",np.nan):>7.4f} '
          f'{r.get("R2_m2",np.nan):>7.4f} '
          f'{r.get("R2_m3",np.nan):>7.4f} '
          f'{r.get("R2_gain_m3_vs_m1",np.nan):>+9.4f} '
          f'{r.get("R2_gain_m3_vs_m2",np.nan):>+9.4f}')
print('-' * 85)

# Group averages
print('\nGroup Averages:')
for grp_val, label in [(1, 'Oil Exporters'), (0, 'Controls')]:
    sub = r2_panel[r2_panel['oil_exporter'] == grp_val]
    if len(sub) == 0: continue
    print(f'  {label} (n={len(sub)}):')
    print(f'    R²_M1={sub["R2_m1"].mean():.4f}  '
          f'R²_M2={sub["R2_m2"].mean():.4f}  '
          f'R²_M3={sub["R2_m3"].mean():.4f}')
    print(f'    Gain M3 vs M1: {sub["R2_gain_m3_vs_m1"].mean():+.4f}  '
          f'Gain M3 vs M2: {sub["R2_gain_m3_vs_m2"].mean():+.4f}')

# --- DiD test on R² improvement ---
print('\n' + '=' * 60)
print('DiD Test: Is R² improvement larger for oil exporters?')
print('=' * 60)

for dep_var, label in [('R2_gain_m3_vs_m1', 'M3 vs M1 (oil jumps vs baseline)'),
                        ('R2_gain_m3_vs_m2', 'M3 vs M2 (oil calib vs generic)')]:
    valid = r2_panel.dropna(subset=[dep_var])
    y = valid[dep_var].values
    X = sm.add_constant(valid['oil_exporter'].astype(float).values)
    reg = sm.OLS(y, X).fit(cov_type='HC1')

    beta = reg.params[1]
    t = reg.tvalues[1]
    p = reg.pvalues[1]
    sig = '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.10 else ''

    print(f'\n  {label}:')
    print(f'    β(oil_exporter) = {beta:+.4f}  t={t:.2f}  p={p:.4f} {sig}')
    print(f'    Interpretation: oil exporters gain {beta:+.4f} more R² from M3')

In [ ]:
# ================================================================
# SCATTER: R² improvement vs oil dependence
# ================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, title in [
    (axes[0], 'R2_gain_m3_vs_m1', 'R² Gain: M3 vs M1 (baseline)'),
    (axes[1], 'R2_gain_m3_vs_m2', 'R² Gain: M3 vs M2 (generic jumps)'),
]:
    valid = r2_panel.dropna(subset=[col])
    oil = valid[valid['oil_exporter'] == 1]
    ctrl = valid[valid['oil_exporter'] == 0]

    ax.scatter(ctrl.index, ctrl[col], color='steelblue',
               label='Controls', alpha=0.7, s=60)
    ax.scatter(oil.index, oil[col], color='tomato',
               label='Oil Exporters', alpha=0.7, s=60, marker='D')

    # Add country labels
    for _, r in valid.iterrows():
        ax.annotate(r['country'][:8], (_, r[col]),
                    fontsize=7, alpha=0.6,
                    xytext=(3, 3), textcoords='offset points')

    ax.axhline(0, color='black', lw=0.8, ls='--')
    ax.set_title(title)
    ax.set_ylabel('R² improvement')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# CROSS-COUNTRY COMPARISON
# ================================================================

rows = []
for ctry, grp in df_res.groupby('country_clean'):
    obs = grp['cds_spread']
    oil = grp['oil_exporter'].iloc[0] if 'oil_exporter' in grp.columns else np.nan
    row = {'country': ctry, 'oil_exporter': oil, 'n_obs': len(grp),
           'avg_cds': obs.mean(), 'avg_scale': grp['scale_mult'].mean()}

    for m, col in [('m1','spread_m1'),('m2','spread_m2'),('m3','spread_m3')]:
        impl = grp[col]
        row[f'rmse_{m}'] = np.sqrt(np.mean((obs - impl)**2))
        row[f'corr_{m}'] = obs.corr(impl)
    rows.append(row)

summary = pd.DataFrame(rows).sort_values('oil_exporter', ascending=False)

# Improvement: positive means M3 is better
summary['rmse_improvement_m3_vs_m1'] = summary['rmse_m1'] - summary['rmse_m3']
summary['rmse_improvement_m3_vs_m2'] = summary['rmse_m2'] - summary['rmse_m3']

print('Cross-Country Model Comparison')
print('=' * 90)
display_cols = ['country', 'oil_exporter', 'avg_cds', 'avg_scale',
                'rmse_m1', 'rmse_m2', 'rmse_m3',
                'rmse_improvement_m3_vs_m1', 'rmse_improvement_m3_vs_m2']
print(summary[display_cols].to_string(index=False, float_format='%.1f'))

# Group averages
if 'oil_exporter' in summary.columns:
    print('\nGroup Averages:')
    group_avg = summary.groupby('oil_exporter')[[
        'rmse_m1', 'rmse_m2', 'rmse_m3',
        'rmse_improvement_m3_vs_m1', 'rmse_improvement_m3_vs_m2'
    ]].mean()
    print(group_avg.round(1).to_string())